# Simulación

1. **Contexto y Sistema Real** El sistema bajo estudio consiste en la infraestructura de servidores on-premise de una compañía tecnológica en fase de expansión. El punto crítico de acceso es el puerto 51413, el cual centraliza el tráfico de servicios y aplicaciones críticas. Actualmente, el sistema opera bajo una arquitectura de balanceo de carga (Load Balancing) que distribuye los flujos entrantes entre múltiples unidades de procesamiento.
2. **Definición del Problema** Ante el crecimiento de la demanda, la empresa enfrenta la necesidad de validar su capacidad de respuesta y determinar si se requiere inversión en hardware adicional o una reconfiguración de la lógica de despacho. El desafío principal es garantizar la Calidad de Servicio (QoS), evitando la saturación de los servidores y minimizando los tiempos de espera. Los paquetes esperan un TTL antes de descartarse. Esto se calcula cuando se les asigna un servidor. Si el servidor atiende hasta 100 flujos actualmente, el paquete calcula que va a llegar a destino a tiempo. Caso contrario, el 30% rechaza la conexión hasta 300 flujos, el 80% rechaza la conexión hasta 500 flujos y si hay más flujos en el servidor, la conexión siempre se va a rechazar. Lo mismo ocurre si una petición espera más de un tiempo preestablecido de Timeout.
3. **Objetivo de la Simulación** Evaluar el comportamiento del sistema mediante la metodología de Evento a Evento (EaE) para determinar la configuración óptima de servidores (Variables de Control $N$ y $M$). Se busca maximizar el throughput y minimizar el tiempo de permanencia en el sistema, analizando tres escenarios: actual, mejor y peor caso.


## Generación de variables + carga de dataset

*Para poder ejecutar correctamente esta sección, primero hay que correr el notebook de **'preparacion_dataset.ipynb'***

In [170]:
import pandas as pd
tabla_de_eventos_futuros = pd.read_csv("./dataset/datasetFinalSimulacion.csv")

tabla_de_eventos_futuros

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol_label,Timestamp,Flow Duration,IntervaloEntreFlujos,IntervaloEntreFlujos_seg
0,131.202.240.87-199.195.249.48-61009-51413-17,131.202.240.87,61009,199.195.249.48,51413,UDP,2015-05-01 11:37:47,64007541,NaN,NaN
1,131.202.240.87-107.191.41.12-61009-51413-17,131.202.240.87,61009,107.191.41.12,51413,UDP,2015-05-01 11:38:51,64049895,0 days 00:01:04,64.0
2,131.202.240.87-74.207.253.79-61009-51413-17,131.202.240.87,61009,74.207.253.79,51413,UDP,2015-05-01 11:38:51,64106424,0 days 00:00:00,0.0
3,131.202.240.87-65.98.72.100-61009-51413-17,131.202.240.87,61009,65.98.72.100,51413,UDP,2015-05-01 11:39:00,54869,0 days 00:00:09,9.0
4,131.202.240.87-79.132.74.152-61009-51413-17,131.202.240.87,61009,79.132.74.152,51413,UDP,2015-05-01 11:39:15,189748,0 days 00:00:15,15.0
...,...,...,...,...,...,...,...,...,...,...
6303,10.152.152.11-103.198.172.77-45213-51413-6,10.152.152.11,45213,103.198.172.77,51413,TCP,2016-02-24 11:33:58,409475,0 days 00:00:00,0.0
6304,10.152.152.11-87.94.140.155-33760-51413-6,10.152.152.11,33760,87.94.140.155,51413,TCP,2016-02-24 11:34:08,4705193,0 days 00:00:10,10.0
6305,10.152.152.11-87.94.140.155-33760-51413-6,10.152.152.11,33760,87.94.140.155,51413,TCP,2016-02-24 11:34:08,4705193,0 days 00:00:00,0.0
6306,10.152.152.11-217.250.238.213-55312-51413-6,10.152.152.11,55312,217.250.238.213,51413,TCP,2016-02-24 11:34:14,1109,0 days 00:00:06,6.0


## Rutina de ingreso a Eventos

### Variables Exógenas:
#### Datos:
* **IA:** Intervalo entre arribos de flujos (según f.d.p. Kappa3 del TP4).
* **TA_UDP:** Tiempo de atención/procesamiento de flujos UDP (según f.d.p. Lomax del TP4).
* **TA_TCP:** Tiempo de atención/procesamiento de flujos TCP (según f.d.p. Lomax del TP4).

#### Variable de Control:
* **N:** Cantidad de Servidores dedicados exclusivamente a UDP (Clase A).
* **M:** Cantidad de Servidores generales para TCP/UDP (Clase B).
* **TO:** Timeout de una conexión.

### Variables Endógenas:
#### Variable de Estado:
* **CU:** Cantidad de flujos UDP en cola. Donde N es límite hasta donde llega i
* **CT:** Cantidad de flujos TCP en cola. Donde M es límite hasta donde llega j
* **CSA:** Cantidad de Servidores Clase A ocupados.
* **CSB:** Cantidad de Servidores Clase B ocupados.

#### Variable de Resultado:
* **PEC_U:** Promedio de espera en cola de los flujos UDP.
* **PEC_T:** Promedio de espera en cola de los flujos TCP.
* **PTO_A (I):** Porcentaje de tiempo ocioso de cada servidor Clase A.
* **PTO_B (I):** Porcentaje de tiempo ocioso de cada servidor Clase B.
* **NTU:** Cantidad total de flujos UDP atendidos.
* **NTT:** Cantidad total de flujos TCP atendidos.
* **PAT:** Porcentaje de Flujos totales arrepentidos TCP respecto a los que no se arrepienten.

In [171]:
class Conexion:
    tps=0.0
    def __init__(self,ip,port,protocolo,ia,flow_duration):
        self.ip=ip
        self.port=port
        self.protocolo=protocolo
        self.ia=ia
        self.flow_duration=flow_duration

### Motor de Arribos y Lógica de Balking (Arrepentimiento)

Esta rutina corresponde al **Evento Futuro No Condicionado (EFNC) de Llegada**. Se encarga de inyectar las entidades al sistema y hacer avanzar el reloj de la simulación mediante la generación del próximo Intervalo de Arribo ($IA$) utilizando la FDP Kappa3 y asignando el Tiempo de Atención ($TA$) con la FDP Lomax obtenidas en el TP4.

**Lógica Estocástica de Arrepentimiento:**
Antes de ingresar, la entidad evalúa la carga actual del servidor consultando las Variables de Estado ($CU + CT + CSA + CSB$). Basado en los límites de tolerancia definidos en la arquitectura del modelo, se genera un número pseudoaleatorio $R \sim U(0,1)$ para determinar probabilísticamente si el flujo ingresa a las colas o es descartado por saturación:
* **$\le 100$ flujos:** Ingreso directo (0% arrepentimiento).
* **$101$ a $300$ flujos:** 30% de probabilidad de rechazo.
* **$301$ a $500$ flujos:** 80% de probabilidad de rechazo.
* **$> 500$ flujos:** Rechazo automático (100%).

Si la entidad no se arrepiente, se determina su protocolo y se la deriva a la rutina de procesamiento de eventos, respetando las Variables de Control ($N$ y $M$).

In [ ]:
import random
from scipy import stats

# Parámetros obtenidos del TP4 (Ajustá estos valores con los que les dio la librería Fitter)
params_kappa3 = {'a': 1.5, 'loc': 0.0, 'scale': 2.5} 
params_lomax = {'c': 2.0, 'loc': 0.0, 'scale': 15.0} 
PROB_UDP = 0.85 # Probabilidad de que el tráfico sea UDP

# Variables globales para control y métricas
contador_llegadas = 0 # Variable auxiliar para imprimir la prueba de escritorio

def generar_evento_llegada(estado):
    global contador_llegadas
    
    # 1. Generación de variables aleatorias (Unificadas a Milisegundos)
    ia_seg = stats.kappa3.rvs(**params_kappa3)
    ia_ms = max(0, ia_seg * 1000)
    
    ta_ms = stats.lomax.rvs(**params_lomax)
    ta_ms = max(0, ta_ms)
    
    # 2. Determinación del protocolo mediante variable aleatoria
    r_tipo = random.random()
    protocolo = 'UDP' if r_tipo <= PROB_UDP else 'TCP'
    
    # IMPORTANTE: El reloj avanza SIEMPRE, programando la próxima llegada en la TEF
    nuevo_tpll = estado.t + ia_ms
    
    # 3. Rutina de Arrepentimiento (Balking)
    servidores_a_ocupados = estado.n - estado.csa.count(None)
    servidores_b_ocupados = estado.m - estado.csb.count(None)
    flujos_totales = len(estado.cu) + len(estado.ct) + servidores_a_ocupados + servidores_b_ocupados
    
    r_arr = random.random()
    arrepentido = False
    
    if flujos_totales <= 100:
        arrepentido = False
    elif 100 < flujos_totales <= 300:
        if r_arr <= 0.30: arrepentido = True
    elif 300 < flujos_totales <= 500:
        if r_arr <= 0.80: arrepentido = True
    else: 
        arrepentido = True
        
    # ---- PRUEBA DE ESCRITORIO (Validación visual para el grupo) ----
    # if contador_llegadas == 0:
    # print(f"| {'Evento':<10} | {'Reloj (T)':<10} | {'TPLL':<10} | {'Protocolo':<10} | {'Flujos Tot.':<12} | {'Acción':<15} |")
    # print("-" * 80)
        
    # if contador_llegadas < 20: # Imprime solo las primeras 20 llegadas para no saturar la consola
    #     accion_str = "Arrepentido" if arrepentido else f"Ingresa {protocolo}"
    #     print(f"| Llegada    | {estado.t:<10.2f} | {nuevo_tpll:<10.2f} | {protocolo:<10} | {flujos_totales:<12} | {accion_str:<15} |")
    contador_llegadas += 1
    # ----------------------------------------------------------------
        
    if arrepentido:
        estado.repentant_flows += 1
        estado.t = nuevo_tpll # Se actualiza el reloj aunque el paquete no ingrese
        estado.tpll = estado.t + ia_ms
        return # Finaliza la ejecución de la rutina
    
    # 4. Inyección de la entidad al sistema
    nueva_conexion = Conexion(ip="192.168.0.1", port=51413, protocolo=protocolo, ia=ia_ms, flow_duration=ta_ms)
    nueva_conexion.tpll = nuevo_tpll
    print(f"Llegada conexion {nueva_conexion.protocolo}:\tIA: {nueva_conexion.ia}, Flow Duration: {nueva_conexion.flow_duration}")
    # Se deriva el paquete a la lógica de procesamiento (Modificación de estado)
    estado.llegada_conexion(nueva_conexion)

## Rutina de procesamiento de evento

Para procesar los eventos, seguiremos la lógica propuesta en la presentación de la temática del tp:

| Evento     | EFNC    | EFC        | Condición                                                                                          |
|------------|---------|------------|----------------------------------------------------------------------------------------------------|
| Llegada    | Llegada | SalidaA (I) | (CU(i) = 1 y CSA <= N)                                                                            |
|            |         | SalidaB (I) | (CU(i) = 1 y CSA = N y CSB < M) OR<br>(CT(j) = 1 y CU(i) = 0 y CSB < M)                        |
| SalidaA (I)| -       | SalidaA (I) | CU(i) > 0                                                                                         |
| SalidaB (I)| -       | SalidaB (I) | Cu(i) >= 2 OR (CT(j) >= 1 y CU(i) <= 1)                                                          |


Según la prioridad, modificaremos las variables de estado

In [ ]:
import sys
class Estado:
    t = 0.0
    tpll=0.0
    ft = 0
    conexiones_procesadas = 0
    conexiones_arrepentidas = 0
    def __init__(self, n, m, to):
        #Registro las variables de control
        self.n = n
        self.m = m
        self.to = to
        #Genero las variables de estado en base a lo anterior
        self.cu = []
        self.ct = []
        self.repentant_flows = 0
        # Para los servidores, generamos una posición por puesto libre que alternamos entre 1 y 0
        self.csa = [None] * n 
        self.tpsa = [sys.float_info.max] * n
        self.csb = [None] * m
        self.tpsb = [sys.float_info.max] * m
        
    def reportar_estado(self):
        print(f"T: {self.t}, TPLL: {self.tpll}\n CT: {len(self.ct)}, CU: {len(self.cu)}\n CSA disponible:{None in self.csa}, TPSA próximo: {min(self.tpsa)}\n CSB disponible:{None in self.csb}, TPSB próximo: {min(self.tpsb)}\nCant conexiones procesadas: {self.conexiones_procesadas}"
             )
        return
        
    def servidor_disponible(self,servidor):
        return (None in servidor)
        
    def ocupar_servidor(self,servidor,conexion):
        try:
            servidor[servidor.index(None)] = conexion
            return

        except:
            print("Servidores llenos. No se puede tomar una conexión en este momento")
            return conexion

    
    def analizar_arrepentimiento(self,conexion):
        pass
        
    def liberar_conexion(self,servidor,tiempo_salida_servidor,index):
        try:
            conexion_saliente = servidor[index]
            servidor[index] = None
            conexion_saliente.tps = tiempo_salida_servidor[index]
            self.tps = self.t + conexion_saliente.flow_duration
            return conexion_saliente

        except:
            print("Servidor vacío. No hay conexión para liberar")

    def determinacion_evento(self):
        #Antes de generar una nueva entrada, procesamos las posibles salidas
        i=0
        for tps in self.tpsa:
            if(tps < self.tpll and self.csa[i] != None):
                self.t = tps
                conexion_saliente = self.liberar_conexion(self.csa,self.tpsa,i)
                
                self.conexiones_procesadas += 1
                if(len(self.cu) >= 1):
                    self.tpsa[i] = self.t + conexion_saliente.flow_duration
                  
                else:
                    self.tpsa[i] = sys.float_info.max
                i += 1
        i=0
        for tps in self.tpsb:
            if(tps < self.tpll and self.csb[i] != None):
                self.t = tps
                conexion_saliente = self.liberar_conexion(self.csb,self.tpsb,i)
                self.conexiones_procesadas += 1
                if(len(self.cu) >= 1 or len(self.ct) >= 1):
                    self.tpsb[i] = self.t + conexion_saliente.flow_duration
                else:
                    self.tpsb[i] = sys.float_info.max
                i += 1
        # Finalmente, generamos la entrada con la fdp
        if(self.tpll <= min(self.tpsa) and self.tpll <= min(self.tpsb)):
            generar_evento_llegada(self)
                

    def llegada_conexion(self, conexion: Conexion):
        #Actualizo las variables llegada la conexión
        self.t = self.tpll
        self.tpll = self.t + conexion.ia
        self.ft += 1
        #Proceso el evento de llegada de una conexión con protocolo UDP
        if(conexion.protocolo == 'UDP'):
            #Ingreso a la cola de espera
            self.cu.append(conexion)
            
            #Si hay un servidor clase A disponible...
            if(self.servidor_disponible(self.csa) and len(self.cu)>0):
                #ocupo servidor con la primera conexión en la cola
                conexion_ingresante = self.cu.pop(0)
                self.ocupar_servidor(self.csa,conexion_ingresante)

                #Calculo el tiempo de salida para esa posición
                self.tpsa[self.csa.index(conexion_ingresante)] = self.t + conexion_ingresante.flow_duration

            #Si hay un servidor clase B disponible...
            if(self.servidor_disponible(self.csb) and len(self.cu)>0):
                #ocupo el servidor
                conexion_ingresante = self.cu.pop(0)
                self.ocupar_servidor(self.csb,conexion_ingresante)
 
                #Calculo el tiempo de salida para esa posición
                self.tpsb[self.csb.index(conexion_ingresante)] = self.t + conexion_ingresante.flow_duration
                
        #Proceso el evento de llegada de una conexión con protocolo TCP   
        if(conexion.protocolo == 'TCP'):
            #Ingreso a la cola de espera
            self.ct.append(conexion)
            
            #Si hay un servidor clase B disponible, y NO HAY UN UDP ESPERANDO...
            if(self.servidor_disponible(self.csb) and len(self.cu) == 0 ):
                #ocupo el servidor
                conexion_ingresante = self.ct.pop(0)
                self.ocupar_servidor(self.csb,conexion_ingresante)
                #Calculo el tiempo de salida para esa posición
                self.tpsb[self.csb.index(conexion_ingresante)] = self.t + conexion_ingresante.flow_duration
            #Si había una conexión UDP esperando, la ingreso
            elif(self.servidor_disponible(self.csb) and len(self.cu) > 0):
                conexion_ingresante = self.cu.pop(0)
                self.ocupar_servidor(self.csb, conexion_ingresante)
                self.tpsb[self.csb.index(conexion_ingresante)] = self.t + conexion_ingresante.flow_duration
       
    def total_time():
        return t
                
    def total_flow_amount():
        return conexiones_procesadas
    

###### Pequeña pruebita de la clase

In [234]:
estado = Estado(1,1,10)
estado.determinacion_evento()
estado.reportar_estado()

Llegada conexion UDP:	IA: 3315.170340693428, Flow Duration: 3.435343468234787
T: 0.0, TPLL: 3315.170340693428
 CT: 0, CU: 0
 CSA disponible:False, TPSA próximo: 3.435343468234787
 CSB disponible:True, TPSB próximo: 1.7976931348623157e+308
Cant conexiones procesadas: 0


In [236]:
estado.determinacion_evento()
estado.reportar_estado()

Llegada conexion UDP:	IA: 9193.188507504607, Flow Duration: 28.612800472285283
T: 4184.330511841999, TPLL: 13377.519019346606
 CT: 0, CU: 0
 CSA disponible:False, TPSA próximo: 4212.943312314284
 CSB disponible:True, TPSB próximo: 1.7976931348623157e+308
Cant conexiones procesadas: 2


In [213]:
estado.determinacion_evento()
estado.reportar_estado()

T: 8763.894639360156, TPLL: 10863.320118253032
 CT: 0, CU: 0
 CSA disponible:False, TPSA próximo: 8768.764001191392
 CSB disponible:True, TPSB próximo: 1.7976931348623157e+308
Cant conexiones procesadas: 2


## Cálculo de variables resultado

In [ ]:
from abc import ABC, abstractmethod

def start_simulation(n, m):
    results = []
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            simulate_flows(i, j, results)
    for result in results:
        result.print_result()
    graph_results(results)

def simulate_flows(n,m, results):
    simulation = Estado(n,m,10)
    for i in range(1, 6000):
        estado.determinacion_evento()
    results.append(AverageUDPIdleTime(n,m).calculate_result(estado.total_udp_idle_time, estado.total_time()))
    results.append(AverageTCPIdleTime(n,m).calculate_result(estado.total_tcp_idle_time, estado.total_time()))
    results.append(AverageUDPWaitTime(n,m).calculate_result(estado.total_udp_leave_time, estado.total_udp_arrival_time, estado.total_udp_service_time, estado.total_flow_amount()))
    results.append(AverageTCPPWaitTime(n,m).calculate_result(estado.total_tcp_leave_time, estado.total_tcp_arrival_time, estado.total_tcp_service_time, estado.total_flow_amount()))
    results.append(AverageUDPServiceTime(n,m).calculate_result(estado.total_udp_leave_time, estado.total_udp_arrival_time, estado.total_flow_amount()))
    results.append(AverageTCPServiceTime(n,m).calculate_result(estado.total_tcp_leave_time, estado.total_tcp_arrival_time, estado.total_flow_amount()))
    results.append(TimeoutFlowsPorcentage(n,m).calculate_result(estado.repentant_flows, estado.total_flow_amount()))

class Result(ABC):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        self.udp_servers_amount = udp_servers_amount
        self.tcp_servers_amount = tcp_servers_amount
        self.description = None
        self.value = None

    @abstractmethod
    def calculate_result(self, *args, **kwargs):
        pass

    def print_result(self):
        print(
            "Para ",
            self.udp_servers_amount,
            " cantidad de servidores UDP y ",
            self.tcp_servers_amount,
            " cantidad de servidores TCP se obtuvo el valor: ",
            self.value,
            " para la variable ",
            self.description,
        )

class AverageIdleTime(Result):
    def calculate_result(self, total_idle_time, total_time):
        self.value = (total_idle_time * 100) / total_time if total_time else 0.0
        return self

class AverageUDPIdleTime(AverageIdleTime):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Tiempo oscioso promedio de los servidores UDP'

class AverageTCPIdleTime(AverageIdleTime):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Tiempo oscioso promedio de los servidores TCP'

class AverageWaitTime(Result):
    def calculate_result(self, total_leave_time, total_arrival_time, total_service_time, total_flow_amount):
        self.value = (total_leave_time - total_arrival_time - total_service_time) / total_flow_amount if total_flow_amount else 0.0
        return self

class AverageUDPWaitTime(AverageWaitTime):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Tiempo promedio de espera de flujos en servidores UDP'

class AverageTCPWaitTime(AverageWaitTime):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Tiempo promedio de espera de flujos en servidores TCP'

class AverageServiceTime(Result):
    def calculate_result(self, total_leave_time, total_arrival_time, total_flow_amount):
        self.value = (total_leave_time - total_arrival_time) / total_flow_amount if total_flow_amount else 0.0
        return self

class AverageUDPServiceTime(AverageServiceTime):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Tiempo promedio de servicio en servidores TCP'

class AverageTCPServiceTime(AverageServiceTime):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Tiempo promedio de servicio en servidores UDP'

class TimeoutFlowsPorcentage(Result):
    def __init__(self, udp_servers_amount, tcp_servers_amount):
        super().__init__(udp_servers_amount, tcp_servers_amount)
        self.description = 'Porcentaje de Flujos cancelados por timeout respecto al total procesado'

    def calculate_result(self, total_timedout_flows, total_flow_amount):
        self.value = (total_timedout_flows * 100) / total_flow_amount if total_flow_amount else 0.0
        return self

In [15]:
import pandas as pd
import matplotlib.pyplot as plt


def graph_results(results):
    data = []
    for r in results:
        if r.value is not None:
            data.append({
                'Result': type(r).__name__,
                'Description': r.description,
                'UDP_Servers': r.udp_servers_amount,
                'TCP_Servers': r.tcp_servers_amount,
                'Value': r.value
            })

    if not data:
        print('No hay resultados para graficar.')
        return

    df = pd.DataFrame(data)
    unique_results = df['Result'].unique()

    for result in unique_results:
        df_Result = df[df['Result'] == result]
        description = df_Result['Description'].iloc[0]

        pivot_table = df_Result.pivot(
            index='UDP_Servers',
            columns='TCP_Servers',
            values='Value'
        )

        best_value = df_Result['Value'].min()
        best_cases = df_Result[df_Result['Value'] == best_value]

        print(f"\n--- Análisis para {result} ---")
        print(f"Mejor valor encontrado: {best_value:.4f}")
        for _, row in best_cases.iterrows():
            print(f"-> Logrado con {row['UDP_Servers']} servidores UDP y {row['TCP_Servers']} servidores TCP.")

        matrix = pivot_table.sort_index().sort_index(axis=1)

        plt.figure(figsize=(10, 8))
        image = plt.imshow(matrix.values, aspect='auto', cmap='YlGnBu')
        plt.colorbar(image, label='Value')
        plt.xticks(range(len(matrix.columns)), matrix.columns)
        plt.yticks(range(len(matrix.index)), matrix.index)

        for row_index in range(matrix.shape[0]):
            for col_index in range(matrix.shape[1]):
                cell_value = matrix.iloc[row_index, col_index]
                if pd.notna(cell_value):
                    plt.text(col_index, row_index, f'{cell_value:.2f}', ha='center', va='center', color='black')

        plt.title(f'{result}\n({description})', fontsize=14, pad=15)
        plt.xlabel('Cantidad de Servidores TCP', fontsize=12)
        plt.ylabel('Cantidad de Servidores UDP', fontsize=12)
        plt.tight_layout()
        plt.show()

Probamos la simulación

In [ ]:
start_simulation(1,1)
start_simulation(2,3)
start_simulation(3,2)
start_simulation(4,1)
start_simulation(1,5)